In [2]:
import pandas as pd
#DATA UNDERSTANDING AND GRAPHS
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns


import pickle

# DATA PREPARATION
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import ast
from collections import Counter

from nltk.util import ngrams
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet


# OUR FUNCTIONS
from utils.visualizations import get_clean_words, extract_words
from langdetect import detect
from utils.functions_eda_preproc import verify_language, clean_text, normalize_text

import os
import pickle
import re
import nltk
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.express as px
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from tqdm import tqdm
from wordcloud import WordCloud
from mittens import GloVe
from gensim.models import Word2Vec
import utils.visualizations as visualizations
import importlib
importlib.reload(visualizations)
from utils.visualizations import word_cloud_generator, show_saved_images,cooccurrence_matrix_sentence_generator, plot_genre_cooccurrence, d2_embeddings_plotter

from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import MultinomialNB
!pip install scikit-learn
from sklearn.naive_bayes import MultinomialNB


from utils.functions_vectors_modelling import evaluate_models, document_vector
import warnings
warnings.filterwarnings('ignore')


In [3]:
train = pd.read_csv('./Datasets/train.csv', index_col = 'id')
test = pd.read_csv('./Datasets/test.csv',index_col = 'id')
train.head()

,title,tag,artist,year,views,features,lyrics
id,,,,,,,
535805,Walk Away,rock,Tony Molina,2013,699,{},When you said you loved me\r\nDid you mean it ...
7519483,Gotta Make It Kid Naruto Rap,rap,Reece Lett,2021,4,{Sl!ck},Kid Naruto Rap\r\n[Hook]\r\nEverybody wants yo...
4892808,​this is what i asked for,pop,Elliot (DNK),2019,389,{},[Verse 1]\r\nPeople tell me I've changed\r\nI ...
1584150,Stealing Hearts,pop,Katie Armiger,2013,126,{},You've been warned about me\r\nDon't try to ge...
7639050,Get Ready,country,John Campbell Munro,1,2,{},[Verse 1]\r\nI can see the end is coming but I...


In [4]:
train = train.rename(columns={'lyrics': 'raw_lyrics'})
test = test.rename(columns={'lyrics': 'raw_lyrics'})

In [5]:
train.drop_duplicates(subset=['raw_lyrics'], keep='first', inplace=True)

train.reset_index(drop=True, inplace=True)
print(f"Train dataset shape after removing duplicates: {train.shape}")

Train dataset shape after removing duplicates: (134789, 7)


In [6]:
train['lyrics_cleaned']= clean_text(train, 'raw_lyrics',
                                      emojis=True, hashtags=True, mentions=True,
                                      urls=True, html_tags=True, square_brackets=True,
                                      newlines=True, punctuation=True, apostrophes=True, whitespaces=True)

In [7]:
test['lyrics_cleaned']= clean_text(test, 'raw_lyrics',
                                      emojis=True, hashtags=True, mentions=True,
                                      urls=True, html_tags=True, square_brackets=True,
                                      newlines=True, punctuation=True, whitespaces=True)

In [8]:
custom_stopwords = ["chorus", "verse", "bridge", "intro", "outro", "hook", "solo", "get",
        "know", "go", "say", "make", "see", "come", "one", "like"]

In [9]:
train['lyrics_normalized'] = train['lyrics_cleaned'].apply(
    lambda x: normalize_text(
        x, 
        custom_stopwords= custom_stopwords,
        tokenized_output=False))

In [10]:
test['lyrics_normalized'] = test['lyrics_cleaned'].apply(
    lambda x: normalize_text(x,
        custom_stopwords= custom_stopwords,
        tokenized_output=False))

In [11]:
X = train["lyrics_normalized"].fillna("").astype(str)
y = train["tag"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training subset: {len(X_train):,} songs")
print(f"Validation subset: {len(X_val):,} songs")

Training subset: 107,831 songs
Validation subset: 26,958 songs


In [12]:
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

def evaluate_models(X_train_vec, X_test_vec, y_train, y_test, representation_name):
    models = {
        'Naive Bayes': MultinomialNB(),
        'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
        'SGD Classifier (SVM)': SGDClassifier(class_weight='balanced', loss='hinge', max_iter=1000, random_state=42),
        #'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=50, random_state=42, n_jobs=-1)
    }
    
    results = []
    
    for name, model in models.items():
        # Verifica valores negativos de forma segura tanto em matrizes esparsas (BoW/TF-IDF) como densas (Word2Vec/GloVe)
        has_negative = (X_train_vec < 0).nnz > 0 if hasattr(X_train_vec, "nnz") else (X_train_vec < 0).any()
        if name == 'Naive Bayes' and has_negative:
            print(f"A saltar {name} (não aceita valores negativos nas representações densas).")
            continue
            
        print(f"Training {name} with {representation_name}...")
        model.fit(X_train_vec, y_train)
        preds = model.predict(X_test_vec)
      
        score = f1_score(y_test, preds, average='macro')
        
        results.append({
            'Representation': representation_name,
            'Model': name,
            'Macro F1-Score': round(score, 4)
        })
        
    return pd.DataFrame(results).sort_values(by='Macro F1-Score', ascending=False)

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import RandomizedSearchCV, train_test_split

# Amostra estratificada de 40% para encontrar a melhor região rapidamente
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=0.40, random_state=42, stratify=y_train
)

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', SGDClassifier(class_weight='balanced', max_iter=1000, random_state=42))
])

# Espaço alargado e justo para todas as famílias (incluindo max_features!)
param_distributions = [
    # 1. SGDClassifier (rápido e escalável)
    {
        'vectorizer': [TfidfVectorizer()],
        'vectorizer__max_features': [15000, 30000, 50000, None],
        'vectorizer__max_df': [0.65, 0.75, 0.85],
        'vectorizer__min_df': [2, 5, 10],
        'vectorizer__ngram_range': [(1, 2), (1, 3)],
        'vectorizer__sublinear_tf': [True, False],
        'classifier': [SGDClassifier(class_weight='balanced', max_iter=1000, random_state=42)],
        'classifier__loss': ['log_loss', 'modified_huber', 'hinge'],
        'classifier__alpha': [0.00001, 0.00005, 0.0001]
    },
    # 2. Logistic Regression (com max_features controlado para não bloquear)
    {
        'vectorizer': [TfidfVectorizer()],
        'vectorizer__max_features': [15000, 30000, 50000],
        'vectorizer__max_df': [0.65, 0.75, 0.85],
        'vectorizer__min_df': [2, 5],
        'vectorizer__ngram_range': [(1, 2), (1, 3)],
        'vectorizer__sublinear_tf': [True, False],
        'classifier': [LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)],
        'classifier__C': [0.5, 1.0, 2.0, 5.0]
    },
    # 3. Naive Bayes (BoW e TF-IDF com alphas baixos para não sobre-suavizar TF-IDF)
    {
        'vectorizer': [CountVectorizer(), TfidfVectorizer()],
        'vectorizer__max_features': [15000, 30000, 50000, None],
        'vectorizer__max_df': [0.65, 0.75, 0.85],
        'vectorizer__min_df': [2, 5, 10],
        'vectorizer__ngram_range': [(1, 2), (1, 3)],
        'classifier': [MultinomialNB()],
        'classifier__alpha': [0.05, 0.1, 0.5, 1.0]
    }
]

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=45,            # Testa 35 combinações diversas em ~2-3 minutos
    scoring='f1_macro',
    cv=3,
    n_jobs=4,
    random_state=42,
    verbose=2
)

random_search.fit(X_train_sample, y_train_sample)

print("Melhor Macro F1-Score (CV):", round(random_search.best_score_, 4))
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

Fitting 3 folds for each of 45 candidates, totalling 135 fits


KeyboardInterrupt: 

In [ ]:
import pandas as pd

cv_df = pd.DataFrame(random_search.cv_results_)
cv_df['Model'] = cv_df['param_classifier'].apply(lambda x: type(x).__name__)
cv_df['Representation'] = cv_df['param_vectorizer'].apply(lambda x: type(x).__name__)

# Tabela estilo mini-testes: melhor score de cada família (Modelo + Vetorizador)
resumo_modelos = (
    cv_df.sort_values('mean_test_score', ascending=False)
    .groupby(['Representation', 'Model'], as_index=False)
    .first()[['Representation', 'Model', 'mean_test_score', 'std_test_score', 'mean_fit_time', 'params']]
    .rename(columns={
        'mean_test_score': 'Melhor Macro F1 (CV)',
        'std_test_score': 'Desvio Padrão',
        'mean_fit_time': 'Tempo Médio (s)'
    })
    .sort_values(by='Melhor Macro F1 (CV)', ascending=False)
    .reset_index(drop=True)
)

resumo_modelos['Melhor Macro F1 (CV)'] = resumo_modelos['Melhor Macro F1 (CV)'].round(4)
resumo_modelos['Desvio Padrão'] = resumo_modelos['Desvio Padrão'].round(4)
resumo_modelos['Tempo Médio (s)'] = resumo_modelos['Tempo Médio (s)'].round(2)

print("=== MELHOR RESULTADO POR MODELO E REPRESENTAÇÃO ===")
display(resumo_modelos)

print("\n=== TOP 10 COMBINAÇÕES TESTADAS ===")
top10 = cv_df.sort_values('mean_test_score', ascending=False)[
    ['Representation', 'Model', 'mean_test_score', 'mean_fit_time', 'params']
].head(10).reset_index(drop=True)
top10['mean_test_score'] = top10['mean_test_score'].round(4)
display(top10)

=== MELHOR RESULTADO POR MODELO E REPRESENTAÇÃO ===


,Representation,Model,Melhor Macro F1 (CV),Desvio Padrão,Tempo Médio (s),params
0,TfidfVectorizer,SGDClassifier,0.5050,0.0057,22.78,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
1,TfidfVectorizer,LogisticRegression,0.4753,0.0009,28.31,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
2,CountVectorizer,MultinomialNB,0.4572,0.0051,21.26,"{'vectorizer__ngram_range': (1, 2), 'vectorize..."
3,TfidfVectorizer,MultinomialNB,0.4204,0.0010,45.79,"{'vectorizer__ngram_range': (1, 3), 'vectorize..."



=== TOP 10 COMBINAÇÕES TESTADAS ===


,Representation,Model,mean_test_score,mean_fit_time,params
0,TfidfVectorizer,SGDClassifier,0.5050,22.776412,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
1,TfidfVectorizer,SGDClassifier,0.5037,39.420347,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
2,TfidfVectorizer,SGDClassifier,0.5022,14.635590,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
3,TfidfVectorizer,SGDClassifier,0.4887,19.078880,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
4,TfidfVectorizer,SGDClassifier,0.4845,25.392283,"{'vectorizer__sublinear_tf': False, 'vectorize..."
5,TfidfVectorizer,SGDClassifier,0.4833,26.653966,"{'vectorizer__sublinear_tf': False, 'vectorize..."
6,TfidfVectorizer,SGDClassifier,0.4818,22.859536,"{'vectorizer__sublinear_tf': False, 'vectorize..."
7,TfidfVectorizer,SGDClassifier,0.4797,22.781456,"{'vectorizer__sublinear_tf': False, 'vectorize..."
8,TfidfVectorizer,SGDClassifier,0.4789,12.179046,"{'vectorizer__sublinear_tf': True, 'vectorizer..."
9,TfidfVectorizer,SGDClassifier,0.4763,47.729798,"{'vectorizer__sublinear_tf': False, 'vectorize..."


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

pipeline_fino = Pipeline([
    ('vectorizer', TfidfVectorizer(max_df=0.75, min_df=5, sublinear_tf=True)),
    ('classifier', SGDClassifier(class_weight='balanced', max_iter=1000, random_state=42))
])

param_grid_fino = {
    'vectorizer__max_features': [40000, 50000, 75000],
    'vectorizer__ngram_range': [(1, 2), (1, 3)],
    'classifier__loss': ['log_loss', 'modified_huber'],
    'classifier__alpha': [0.00002, 0.00005, 0.00008]
}

grid_fino = GridSearchCV(
    estimator=pipeline_fino,
    param_grid=param_grid_fino,
    scoring='f1_macro',
    cv=3,
    n_jobs=4
)

grid_fino.fit(X_train_sample, y_train_sample)

print("Melhor Macro F1-Score (CV Fino):", round(grid_fino.best_score_, 4))
for param, value in grid_fino.best_params_.items():
    print(f"  {param}: {value}")

# Top 5 combinações da grelha fina
top5_fino = (
    pd.DataFrame(grid_fino.cv_results_)
    .sort_values('mean_test_score', ascending=False)[['mean_test_score', 'std_test_score', 'params']]
    .head(5)
    .reset_index(drop=True)
)
top5_fino['mean_test_score'] = top5_fino['mean_test_score'].round(4)
top5_fino['std_test_score'] = top5_fino['std_test_score'].round(4)
display(top5_fino)

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

pipeline_fino = Pipeline([
    ('vectorizer', TfidfVectorizer(sublinear_tf=True)),
    ('classifier', SGDClassifier(class_weight='balanced', max_iter=1000, random_state=42))
])

# Vizinhança fina à volta de TODOS os parâmetros vencedores
param_grid_fino = {
    'vectorizer__max_df': [0.70, 0.75, 0.80],
    'vectorizer__min_df': [4, 5, 6],
    'vectorizer__max_features': [40000, 50000, 75000],
    'vectorizer__ngram_range': [(1, 2), (1, 3)],
    'classifier__loss': ['log_loss', 'modified_huber'],
    'classifier__alpha': [0.00003, 0.00005, 0.00007]
}

grid_fino = GridSearchCV(
    estimator=pipeline_fino,
    param_grid=param_grid_fino,
    scoring='f1_macro',
    cv=3,
    n_jobs=3  # 3 workers mantêm o PC rápido e a RAM segura
)

grid_fino.fit(X_train_sample, y_train_sample)

print("Melhor Macro F1-Score (CV Fino):", round(grid_fino.best_score_, 4))
for param, value in grid_fino.best_params_.items():
    print(f"  {param}: {value}")

# Top 5 combinações da grelha fina para veres se houve diferença real
top5_fino = (
    pd.DataFrame(grid_fino.cv_results_)
    .sort_values('mean_test_score', ascending=False)[['mean_test_score', 'std_test_score', 'params']]
    .head(5)
    .reset_index(drop=True)
)
top5_fino['mean_test_score'] = top5_fino['mean_test_score'].round(4)
top5_fino['std_test_score'] = top5_fino['std_test_score'].round(4)
display(top5_fino)

Melhor Macro F1-Score (CV Fino): 0.5085
  classifier__alpha: 3e-05
  classifier__loss: log_loss
  vectorizer__max_df: 0.7
  vectorizer__max_features: 75000
  vectorizer__min_df: 5
  vectorizer__ngram_range: (1, 2)


,mean_test_score,std_test_score,params
0,0.5085,0.0066,"{'classifier__alpha': 3e-05, 'classifier__loss..."
1,0.5085,0.0066,"{'classifier__alpha': 3e-05, 'classifier__loss..."
2,0.5085,0.0066,"{'classifier__alpha': 3e-05, 'classifier__loss..."
3,0.5082,0.0061,"{'classifier__alpha': 3e-05, 'classifier__loss..."
4,0.5082,0.0061,"{'classifier__alpha': 3e-05, 'classifier__loss..."


In [ ]:
import os
import pandas as pd
from sklearn.metrics import f1_score, classification_report

# Se num notebook futuro saltares a Célula 1, troca 'grid_fino' por 'random_search' nas 2 linhas abaixo
search_vencedor = grid_fino
best_model = search_vencedor.best_estimator_

# 1. Treinar o modelo vencedor nos 100% do X_train e prever no X_val
best_model.fit(X_train, y_train)
val_preds = best_model.predict(X_val)

val_f1 = f1_score(y_val, val_preds, average='macro')
print(f"Macro F1-Score Global no X_val (100% treino): {val_f1:.4f}\n")

# 2. Tabela com o F1-Score de cada género
report = classification_report(y_val, val_preds, output_dict=True)
df_generos = pd.DataFrame([
    {"Género": g, "F1-Score": round(report[g]["f1-score"], 4)}
    for g in sorted(y_val.unique())
]).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
display(df_generos)

# 3. Guardar no ficheiro CSV comparativo
linha = pd.DataFrame([{
    "Teste": "sem_duplicados_grid_fino",  # <-- muda este nome em cada experiência
    "Macro_F1": round(val_f1, 4),
    "country": round(report["country"]["f1-score"], 4),
    "rb": round(report["rb"]["f1-score"], 4),
    "pop": round(report["pop"]["f1-score"], 4),
    "rock": round(report["rock"]["f1-score"], 4),
    "misc": round(report["misc"]["f1-score"], 4),
    "rap": round(report["rap"]["f1-score"], 4),
    "Params": str(search_vencedor.best_params_)
}])

ficheiro_csv = "meus_resultados.csv"
linha.to_csv(ficheiro_csv, mode="a", header=not os.path.exists(ficheiro_csv), index=False)
display(pd.read_csv(ficheiro_csv).sort_values(by="Macro_F1", ascending=False))

Macro F1-Score Global no X_val (100% treino): 0.5203



,Género,F1-Score
0,rap,0.8553
1,pop,0.6530
2,misc,0.5669
3,rock,0.4215
4,country,0.3211
5,rb,0.3038


,Teste,Macro_F1,country,rb,pop,rock,misc,rap,Params
1,sem_duplicados_grid_fino,0.5203,0.3211,0.3038,0.6530,0.4215,0.5669,0.8553,"{'classifier__alpha': 3e-05, 'classifier__loss..."
0,teste_só_remover_duplicados,0.5081,0.2873,0.2647,0.6555,0.4138,0.5845,0.8426,{'classifier': SGDClassifier(class_weight='bal...


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels = sorted(y_val.unique())
cm = confusion_matrix(y_val, val_preds, labels=labels)
cm_norm = confusion_matrix(y_val, val_preds, labels=labels, normalize='true') * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Contagens absolutas
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Contagens)', fontweight='bold')
axes[0].set_xlabel('Previsto')
axes[0].set_ylabel('Real')

# 2. Percentagem por género real (mais fácil de ver confusões em classes pequenas)
sns.heatmap(
    cm_norm, annot=True, fmt='.1f', cmap='Oranges',
    xticklabels=labels, yticklabels=labels, ax=axes[1]
)
axes[1].set_title('Confusion Matrix (% por Género Real)', fontweight='bold')
axes[1].set_xlabel('Previsto')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

NameError: name 'y_val' is not defined